# Project 4: Financial Market Analysis
Analyze stock price trends, risk-return statistics, and construct an optimized investment portfolio.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

df = pd.read_csv("datasets/financial_market_data.csv")
df["Date"] = pd.to_datetime(df["Date"])
print("Dataset Loaded. Shape:", df.shape)
df.head()

Dataset Loaded. Shape: (3570, 8)


,Date,Ticker,Sector,Open,High,Low,Close,Volume
0,2022-01-03,AAPL,Technology,150.00,151.04,148.74,150.49,10667852
1,2022-01-04,AAPL,Technology,150.49,155.47,149.73,155.07,6419860
2,2022-01-05,AAPL,Technology,155.07,155.20,151.02,151.54,6792714
3,2022-01-06,AAPL,Technology,151.54,152.30,148.58,149.53,12167214
4,2022-01-07,AAPL,Technology,149.53,152.93,146.71,151.87,9077336


## 1. Close Price Trends & Moving Averages
Visualize the daily closing prices of AAPL, MSFT, GOOGL, AMZN, and TSLA, showing a bullish trend for the tech sector.

In [2]:
plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x='Date', y='Close', hue='Ticker', linewidth=1.5)
plt.title("Stock Closing Prices (2022 - 2025)")
plt.ylabel("Stock Price ($)")
plt.xlabel("Date")
plt.legend(title="Stock Ticker")
plt.tight_layout()
plt.savefig("visualizations/stock_trends.png")
plt.savefig("../visualizations/stock_trends.png")
plt.show()

C:\Users\HARISH RAGHAVENDER\AppData\Local\Temp\ipykernel_19100\1234487978.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Daily Returns & Correlation Analysis
Calculate daily returns and visualize correlations between stocks.

In [3]:
# Pivot closing prices
pivot_df = df.pivot(index='Date', columns='Ticker', values='Close')
returns_df = pivot_df.pct_change().dropna()

# Correlation Matrix
corr = returns_df.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", vmin=0, vmax=1)
plt.title("Daily Stock Returns Correlation Matrix")
plt.tight_layout()
plt.savefig("visualizations/correlation_matrix.png")
plt.savefig("../visualizations/correlation_matrix.png")
plt.show()

C:\Users\HARISH RAGHAVENDER\AppData\Local\Temp\ipykernel_19100\2024110151.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Risk vs. Return Profile
Compare annual returns against volatility (standard deviation of daily returns) for each stock.

In [4]:
# Annualized returns and volatility (252 trading days)
stats_df = pd.DataFrame({
    'Annualized Return (%)': returns_df.mean() * 252 * 100,
    'Annualized Volatility (%)': returns_df.std() * np.sqrt(252) * 100
})

plt.figure(figsize=(8, 6))
sns.scatterplot(data=stats_df, x='Annualized Volatility (%)', y='Annualized Return (%)', hue=stats_df.index, s=200, palette='Set1')
for i, txt in enumerate(stats_df.index):
    plt.annotate(txt, (stats_df['Annualized Volatility (%)'].iloc[i]+0.5, stats_df['Annualized Return (%)'].iloc[i]+0.5), fontsize=12)

plt.title("Risk vs. Return Profile (Annualized)")
plt.xlabel("Volatility (Risk, %)")
plt.ylabel("Return (Annualized, %)")
plt.tight_layout()
plt.savefig("visualizations/risk_return.png")
plt.savefig("../visualizations/risk_return.png")
plt.show()

print(stats_df)

        Annualized Return (%)  Annualized Volatility (%)
Ticker                                                  
AAPL                39.022756                  22.550052
AMZN                18.807258                  28.381091
GOOGL               33.352596                  26.237264
MSFT                29.458141                  20.769280
TSLA                46.621573                  46.541998


C:\Users\HARISH RAGHAVENDER\AppData\Local\Temp\ipykernel_19100\1293413641.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Portfolio Optimization (Efficient Frontier)
Simulate 5,000 random portfolios of these 5 stocks to determine the Efficient Frontier using Mean-Variance Optimization.

In [5]:
# Mean returns and covariance matrix
mean_returns = returns_df.mean()
cov_matrix = returns_df.cov()
num_portfolios = 5000
results = np.zeros((3, num_portfolios))
weights_record = []

for i in range(num_portfolios):
    weights = np.random.random(len(pivot_df.columns))
    weights /= np.sum(weights)
    weights_record.append(weights)
    
    portfolio_return = np.sum(mean_returns * weights) * 252
    portfolio_std_dev = np.sqrt(np.dot(weights.T, np.dot(cov_matrix * 252, weights)))
    
    results[0,i] = portfolio_std_dev
    results[1,i] = portfolio_return
    # Sharpe Ratio (Assuming 0% risk free rate)
    results[2,i] = portfolio_return / portfolio_std_dev

# Max Sharpe Ratio Portfolio
max_sharpe_idx = np.argmax(results[2])
sd_max_sharpe, ret_max_sharpe = results[0,max_sharpe_idx], results[1,max_sharpe_idx]
best_weights = weights_record[max_sharpe_idx]

# Min Volatility Portfolio
min_vol_idx = np.argmin(results[0])
sd_min_vol, ret_min_vol = results[0,min_vol_idx], results[1,min_vol_idx]

plt.figure(figsize=(10, 6))
sc = plt.scatter(results[0]*100, results[1]*100, c=results[2], cmap='viridis', marker='o', s=10, alpha=0.3)
plt.colorbar(sc, label='Sharpe Ratio')
plt.scatter(sd_max_sharpe*100, ret_max_sharpe*100, marker='*', color='r', s=200, label='Max Sharpe Ratio')
plt.scatter(sd_min_vol*100, ret_min_vol*100, marker='X', color='g', s=200, label='Min Volatility')
plt.title("Efficient Frontier - Modern Portfolio Theory")
plt.xlabel("Portfolio Volatility (Risk, %)")
plt.ylabel("Portfolio Return (Expected, %)")
plt.legend()
plt.tight_layout()
plt.savefig("visualizations/efficient_frontier.png")
plt.savefig("../visualizations/efficient_frontier.png")
plt.show()

print("Max Sharpe Ratio Allocation:")
for ticker, w in zip(pivot_df.columns, best_weights):
    print(f"{ticker}: {w*100:.2f}%")
print(f"Expected Annual Return: {ret_max_sharpe*100:.2f}%")
print(f"Annualized Volatility: {sd_max_sharpe*100:.2f}%")
print(f"Sharpe Ratio: {results[2, max_sharpe_idx]:.2f}")

Max Sharpe Ratio Allocation:
AAPL: 32.68%
AMZN: 10.14%
GOOGL: 19.28%
MSFT: 30.83%
TSLA: 7.08%
Expected Annual Return: 33.47%
Annualized Volatility: 11.81%
Sharpe Ratio: 2.83


C:\Users\HARISH RAGHAVENDER\AppData\Local\Temp\ipykernel_19100\4094753982.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
